# LevelForge — Adversarial level evolution (UED) on Colab

Runs the full experiment: **PPO agent** × **four curriculum conditions** (GA adversary, random search, random levels, fixed set) × **3 seeds**, the **frozen-agent search experiment**, and the **analysis** (tables, permutation tests, figures).

**Runtime:** `Runtime → Change runtime type → T4 GPU` (not TPU — PyTorch here only uses CUDA or CPU).

Results are written straight to Google Drive and every finished run is skipped on re-run, so a disconnect only loses the run in progress — just re-run the cells.

In [ ]:
# 1 — Mount Drive and extract the project zip (upload LevelForge.zip to MyDrive first)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, zipfile
shutil.rmtree('/content/levelforge', ignore_errors=True)
with zipfile.ZipFile('/content/drive/MyDrive/LevelForge.zip') as z:
    z.extractall('/content/levelforge')
root = next(os.path.join(dp) for dp, dn, fn in os.walk('/content/levelforge') if 'pyproject.toml' in fn)
os.chdir(root); print('project root:', root)

In [ ]:
# 2 — Dependencies (numpy/torch/matplotlib are preinstalled on Colab)
!pip -q install "stable-baselines3>=2.3" "gymnasium>=0.29"
import os, torch
os.environ['PYTHONPATH'] = 'src'
print('CUDA available:', torch.cuda.is_available())
EXP = '/content/drive/MyDrive/levelforge_runs/ued_7x7'   # results persist on Drive
CFG = 'configs/ued_7x7.json'

In [ ]:
# 3 — Test suite (37 tests, ~15 s) and a 3-minute end-to-end smoke run
!python -m pytest -q tests
!python -m levelforge ued-run --exp runs/smoke --config configs/ued_smoke.json --seeds 0 --condition ga

## Why PPO? (agent comparison)
Trains the original from-scratch DQN and PPO for the same number of environment steps on the same levels. Supports the agent choice in the report with data.

In [ ]:
!python -m levelforge agent-bench --exp $EXP --config $CFG --steps 400000 --seeds 0

## Main experiment
Pretraining is shared by all four conditions of a seed (same starting agent). Each cell is resumable.

In [ ]:
!python -m levelforge ued-pretrain --exp $EXP --config $CFG --seeds 0 1 2

In [ ]:
!python -m levelforge ued-run --exp $EXP --config $CFG --seeds 0 1 2 --condition ga

In [ ]:
!python -m levelforge ued-run --exp $EXP --config $CFG --seeds 0 1 2 --condition random_search

In [ ]:
!python -m levelforge ued-run --exp $EXP --config $CFG --seeds 0 1 2 --condition random

In [ ]:
!python -m levelforge ued-run --exp $EXP --config $CFG --seeds 0 1 2 --condition fixed

## Failure-discovery efficiency (frozen agent)
GA vs random search against the pretrained agent, 1024 candidate levels each, 3 seeds.

In [ ]:
!python -m levelforge ued-attack --exp $EXP --config $CFG --seeds 0 1 2 --budget 1024

## Analysis — tables, significance tests, figures

In [ ]:
!python -m levelforge ued-analyze --exp $EXP
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{EXP}/analysis/results.md').read()))
for f in ['success_curves', 'curriculum_regret', 'attack_efficiency']:
    p = f'{EXP}/analysis/{f}.png'
    if os.path.exists(p): display(Image(p))

In [ ]:
# Look at some evolved levels: the hardest levels the GA added in the last cycle (seed 0)
import json, glob
from levelforge.grid import Level
last = sorted(glob.glob(f'{EXP}/ga_seed0/levels/cycle_*.json'))[-1]
for d in json.load(open(last))[:4]:
    print(Level.from_dict(d).to_ascii(), '
')

## Notes
- More seeds make the statistics stronger: with 3 seeds per condition the smallest possible permutation p-value is 0.10; with 5 seeds it is ~0.008. Add seeds with `--seeds 3 4` (finished runs are skipped).
- Harder setting (2 boxes) for an extension: copy `configs/ued_7x7.json`, set `"n_boxes": 2` and `"pretrain_steps": 1500000`, use a new `EXP` folder.